# 4. Equilibration with GROMACS

We put an SDS micelle (a ball) in water, inside a project, and equilibrate it with GROMACS. The equilibration runs in stages:

- **stage 0**: energy minimisation, about 1–2 minutes on Colab
- **stage 1**: short MD with position restraints, about 20 minutes on Colab. We start it in the background, watch it and cancel it.

Then we look at the outputs, save the project and download it, to open in VMD.

Run the setup cell first. On Colab it installs GROMACS and Shapespyer, which takes about 3 minutes. To keep the project after the Colab runtime ends, tick `SAVE_PROJECTS_TO_DRIVE` before you run it: Colab then asks you once to sign in to Google Drive.

In [ ]:
# @title Setup: on Colab, install GROMACS and Shapespyer (about 3 min)
SAVE_PROJECTS_TO_DRIVE = False  # @param {type:"boolean"}

import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files

if IN_COLAB:
    !apt-get -qq update > /dev/null && apt-get -qq install -y --no-install-recommends gromacs > /dev/null
    %pip install -q "shapespyer==0.3.0"

    # nglview, behind .show(), draws molecules through Colab's widget manager
    from google.colab import output
    output.enable_custom_widget_manager()

    from shapes.project import Config
    config = Config(CONFIG_FILE)
    # the library and Temp stay on the fast local disk; /content shows in the Files panel
    config.app_path = "/content/shapespyer"
    # projects go there too, or to Drive, where they stay after the runtime ends
    config.projects_path = None
    if SAVE_PROJECTS_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")  # asks once for Google sign-in
        config.projects_path = "/content/drive/MyDrive/shapespyer/Projects"
    config.save()
elif not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; open the notebook from the shapespyer-workshop folder"
    raise FileNotFoundError(err_msg)

from shapes.project import Config
from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
config = AppFileManager(CONFIG_FILE).config
if config.projects_path != Config(CONFIG_FILE).projects_path:
    err_msg = "Shapespyer already runs with other folders: Runtime > Restart session, then run this cell again"
    raise RuntimeError(err_msg)
print("Shapespyer files:", config.app_path, "| projects:", config.projects_path)
if config.gmx_problem is not None:
    err_msg = (
        f"This notebook needs GROMACS: {config.gmx_problem}. Install it, or set gmx_executable"
        f" in {CONFIG_FILE}. On Windows, the easy way is to run this notebook in WSL or on Colab."
    )
    raise RuntimeError(err_msg)
print("GROMACS:", config.gmx_version)

In [ ]:
import re
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj
import nglview
import numpy as np

from shapes.basics.utils import LogConfiguration
from shapes.project import Ball, Ion, Library, Project, Surfactant
from shapes.project.analysis import AnalysisConfiguration
from shapes.project.species import Solvent

LogConfiguration("WARNING")

library = Library()
library.clean_temp()

# a new folder each run, so a project saved before, such as one in Drive, is kept
DEMO_PATH = config.projects_path / f"DemoProj-{time.strftime('%Y%m%d-%H%M%S')}"

### Create the project and its sample model

The project goes into the projects folder that the setup cell printed: `.data/Projects` locally, `/content/shapespyer/Projects` on Colab, or your Drive. The path given to `Project` is only noted: the folder is made on the first save.

SDS is charged, so it gets sodium (NAT) as its counter ion.

In [ ]:
proj = Project(path=DEMO_PATH)
model = proj.sample_model
model.name = "SDS_ball"

ball = Ball(name="SDS_micelle")
ball.layers[0].dmin = 0.5
ball.layers[0].rmin = 1.2

sds = Surfactant.load("SDS")
sds.mint = 41
sds.counter_ion = Ion.load("NAT")
ball.components.append(sds)

model.structure = ball
model.buffer.solvent = Solvent.load("TIP3")

### Assemble the sample model

The assembly builds the ball, puts in one Na⁺ per SDS with `gmx insert-molecules`, and fills the box with water with `gmx solvate`. The files are made in Temp, in the application directory.

In [ ]:
model.assemble()
for path in model.paths:
    print(path)

These are the GROMACS commands the assembly ran. Its log holds their full output too:

In [ ]:
print("\n".join(line for line in model.job.log.read_text().splitlines() if line.startswith("$ ")))

### Set up the equilibration

Stage *k* runs `equil<k>.mdp`, starting from the final `.gro` of stage *k − 1*. The library's CHARMM36 set has 7 stages:

In [ ]:
equilibration = proj.analysis.equilibration
equilibration.configuration = AnalysisConfiguration.load("CHARMM36")

for mdp in equilibration.configuration.paths:
    settings = re.findall(r"^\s*(integrator|nsteps|dt)\s*=\s*(\S+)", mdp.read_text(), re.MULTILINE)
    print(mdp.name, dict(settings))

### Stage 0: energy minimisation

`first_stage` and `last_stage` choose the stages to run. `.run()` waits until GROMACS ends.

In [ ]:
equilibration.first_stage = 0
equilibration.last_stage = 0

start = time.perf_counter()
equilibration.run(model)
print(f"stage 0 took {time.perf_counter() - start:.0f} s")

for path in equilibration.stages[0].paths:
    print(path)

The files of a stage:

- `.tpr`: the run input, made by `gmx grompp`
- `.log`: the log of `gmx mdrun`
- `.edr`: energies
- `.trr`: trajectory (`.xtc` for MD stages)
- `.gro`: final coordinates; the next stage starts from them

Did the minimisation converge?

In [ ]:
eq0 = {path.name: path for path in equilibration.stages[0].paths}
eq0_log = eq0[f"{model.name}-eq0.log"]
for line in eq0_log.read_text().splitlines():
    if "converged" in line or "Potential Energy" in line or "Maximum force" in line:
        print(line)

The minimised micelle, without the water:

In [ ]:
frame = mdtraj.load(eq0[f"{model.name}-eq0.gro"])
# mdtraj renames TIP3 to HOH on loading, so select it with the "water" keyword
shown = frame.atom_slice(frame.top.select("not water"))
print(frame.n_atoms, "atoms,", shown.n_atoms, "without the water")
nglview.show_mdtraj(shown)

### Stage 1 in the background

`background=True` makes `.run()` return at once, while GROMACS goes on. Meanwhile we can check if it still runs, which stage it is on, what GROMACS prints and which files it writes.

In [ ]:
equilibration.first_stage = 1
equilibration.last_stage = 1
equilibration.run(model, background=True)

Is it running, and which stage? Run this cell again to update it.

In [ ]:
print("running:", equilibration.files_locked, "| stage:", equilibration.stage)

`gmx mdrun` writes its progress to the job's log. Run this cell again: the step goes up.

In [ ]:
time.sleep(15)  # give mdrun a moment to start
progress = equilibration.job.log.read_text().replace("\r", "\n").splitlines()
print("\n".join(line for line in progress[-5:] if line.strip()))

The files being written are in the job's scratch folder, in Temp. Run this cell again: they grow.

In [ ]:
scratch = equilibration.job.scratch
print(scratch)
for path in sorted(scratch.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size:>12,} B  {path.relative_to(scratch)}")

### Cancel

`cancel()` stops GROMACS. The scratch with the unfinished stage 1 is deleted. Stage 0 keeps its files.

In [ ]:
equilibration.job.cancel()
print("error:", equilibration.job.error)
print("running:", equilibration.files_locked)
print("scratch left:", scratch.exists())
for stage in equilibration.stages:
    print(stage.name, [path.name for path in stage.paths])

### If there is time: run stage 1 to the end

About 20 minutes on Colab. Tick the box (on Colab) or set `True`, then run the cell. A run that ends is published to `equilibration.stages[1]`.

In [ ]:
RUN_STAGE_1 = False  # @param {type:"boolean"}

if RUN_STAGE_1:
    start = time.perf_counter()
    equilibration.run(model)
    print(f"stage 1 took {time.perf_counter() - start:.0f} s")
    for path in equilibration.stages[1].paths:
        print(path)

The stage 1 trajectory, without the water:

In [ ]:
if RUN_STAGE_1:
    eq1 = {path.name: path for path in equilibration.stages[1].paths}
    trajectory = mdtraj.load(eq1[f"{model.name}-eq1.xtc"], top=eq1[f"{model.name}-eq1.gro"])
    view = nglview.show_mdtraj(trajectory.atom_slice(trajectory.top.select("not water")))
else:
    view = "Stage 1 wasn't run"
view

How big is the micelle during stage 1? The radius of gyration (Rg), with the real masses of the atoms, gives one value per frame. If Rg stays about the same, the micelle keeps its size.

Then we save the micelle alone, without the water. These files are small, to download and open in VMD: `vmd micelle-eq1.gro micelle-eq1.xtc`.

In [ ]:
if RUN_STAGE_1:
    micelle = trajectory.atom_slice(trajectory.top.select("resname SDS"))
    masses = np.array([atom.element.mass for atom in micelle.top.atoms])
    rg = mdtraj.compute_rg(micelle, masses=masses)  # nm, one value per frame
    print(f"{micelle.n_frames} frames, Rg from {rg.min():.2f} to {rg.max():.2f} nm")

    plt.plot(micelle.time, rg)
    plt.xlabel("time (ps)")
    plt.ylabel("Rg of the micelle (nm)")
    plt.show()

    micelle.save_xtc("micelle-eq1.xtc")
    micelle[-1].save_gro("micelle-eq1.gro")  # last frame, also the topology for VMD
    if IN_COLAB:
        from google.colab import files
        files.download("micelle-eq1.xtc")
        files.download("micelle-eq1.gro")

### Save the project

`save()` makes the project folder and copies into it the sample model, the analysis and every asset in them, with the stages. `ser_data.yaml` records the whole project, and `manifest.yaml` lists the files.

In [ ]:
proj.save()
for path in sorted(proj.path.rglob("*")):
    if path.is_file():
        print(path.relative_to(proj.path))

`Project.load` restores it from the folder, with the stages:

In [ ]:
loaded = Project.load(DEMO_PATH)
for stage in loaded.analysis.equilibration.stages:
    print(stage.name, [path.name for path in stage.paths])

### Download and open in VMD

On Colab, this zips the project and downloads it. Unzip it on your computer. Each stage has its own folder, `Stage/eq<k>-<id>/`. Open a stage in VMD from its folder, for example:

```
vmd SDS_ball-eq0.gro
vmd SDS_ball-eq1.gro SDS_ball-eq1.xtc
```

In [ ]:
archive = shutil.make_archive("DemoProj", "zip", proj.path)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
archive

### Clean up (optional)

This deletes the saved project, the zip, the micelle files and Temp, where the model and the stages were made. `force=True` is needed, because the assets of this session still hold their files there. Skip it to keep the project, for example in Drive.

In [ ]:
project_dir = proj.path
library.delete_project(project_dir, force=True)
Path(archive).unlink(missing_ok=True)
for path in Path().glob("micelle-eq1.*"):
    path.unlink()
library.clean_temp(force=True)
print(project_dir.exists())